# 02 — Lemmy Moderation Log Collection

Collects public moderation logs from three Lemmy instances:
- **lemmy.ml** — tech-focused, strict moderation
- **lemmy.world** — general, largest instance
- **beehaw.org** — community-first, explicit moderation guidelines

Data is stored in `../data/moderation.db` tables:
- `lemmy_posts` — post content (title + body)
- `lemmy_modlog` — removal actions, reasons, moderator names

No authentication required — modlogs are public per Lemmy's design.

In [ ]:
import sys
sys.path.insert(0, "..")

import sqlite3
import pandas as pd
from src.lemmy_scraper import get_conn, setup_db, INSTANCES

print("Target instances:", INSTANCES)

## 1. Connectivity check

In [ ]:
import requests

for instance in INSTANCES:
    try:
        resp = requests.get(
            f"{instance}/api/v3/modlog",
            params={"limit": 1, "type_": "ModRemovePost"},
            headers={"Accept": "application/json"},
            timeout=10
        )
        print(f"  {instance:<30} → HTTP {resp.status_code}")
    except Exception as e:
        print(f"  {instance:<30} → ERROR: {e}")

## 2. Database setup

In [ ]:
setup_db()
print("✓ Schema ready")

## 3. Collect moderation logs

In [ ]:
from src.lemmy_scraper import collect_all_instances

total_posts, total_logs = collect_all_instances()
print(f"\nTotal posts stored   : {total_posts}")
print(f"Total modlog entries : {total_logs}")

## 4. Row counts

In [ ]:
conn = get_conn()
print("── Table sizes ──────────────────────────────")
for t in ["lemmy_posts", "lemmy_modlog"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<22}: {n:>6} rows")
conn.close()

## 5. Sample modlog records

In [ ]:
conn = get_conn()
df = pd.read_sql_query("""
    SELECT
        m.instance,
        SUBSTR(p.title, 1, 80) AS title_preview,
        m.mod_action,
        m.reason,
        m.removed,
        m.actioned_at
    FROM lemmy_modlog m
    LEFT JOIN lemmy_posts p ON m.post_id = p.post_id AND m.instance = p.instance
    WHERE m.reason IS NOT NULL AND m.reason != ''
    ORDER BY m.actioned_at DESC
    LIMIT 10
""", conn)
conn.close()

pd.set_option("display.max_colwidth", 85)
df

## 6. Removal reason distribution

In [ ]:
conn = get_conn()
df_reasons = pd.read_sql_query("""
    SELECT
        instance,
        reason,
        COUNT(*) AS count
    FROM lemmy_modlog
    WHERE reason IS NOT NULL AND reason != ''
    GROUP BY instance, reason
    ORDER BY count DESC
    LIMIT 20
""", conn)
conn.close()
print(df_reasons.to_string(index=False))